# Aula 2 · EDA, regressão e classificação — Lab Python
**Prof.ª Fabiana · Fundamentos de IA**

Vamos trabalhar com o conjunto **Palmer Penguins** (344 pinguins, 3 espécies). Tudo é implementado **com NumPy**, para que a álgebra linear dos slides fique visível; usamos `pandas` só para carregar e explorar os dados.

| Parte | Tema |
|---|---|
| 1 | Análise exploratória (EDA) |
| 2 | Divisão treino/teste e normalização |
| 3 | Regressão linear: equações normais, gradiente, métricas, ridge |
| 4 | Classificação: regressão logística, métricas, ROC, softmax e k-NN |

Complete as células marcadas com `# TODO`. Os `assert` conferem as respostas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

URL = "https://iegawa.github.io/fundamentos-ia/aula2/penguins.csv"
df = pd.read_csv(URL)
df.head()

## Parte 1 · Análise exploratória

**1.1** Quantas linhas e colunas? Quantos valores ausentes por coluna? Quantos pinguins de cada espécie?

In [ ]:
# TODO: sua implementação aqui
raise NotImplementedError

**1.2** Remova as linhas sem medidas numéricas (ausentes em `bill_length_mm`). Mantenha os pinguins sem `sex`, pois não usaremos essa coluna. Guarde o resultado em `d` e confira que sobram 342 linhas.

In [ ]:
num = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
# TODO: sua implementação aqui
raise NotImplementedError
assert len(d) == 342
d[num].describe()

**1.3 Estatística em forma matricial.** Com $\mathbf X$ = as 4 colunas numéricas, calcule **sem usar `pandas`**:
$\boldsymbol\mu=\tfrac1n\mathbf X^\top\mathbf 1$, $\tilde{\mathbf X}=\mathbf X-\mathbf 1\boldsymbol\mu^\top$, $\boldsymbol\Sigma=\tfrac{1}{n-1}\tilde{\mathbf X}^\top\tilde{\mathbf X}$ e a matriz de correlação $\mathbf R=\mathbf D^{-1}\boldsymbol\Sigma\mathbf D^{-1}$, com $\mathbf D=\operatorname{diag}(\sqrt{\Sigma_{jj}})$.

In [ ]:
X = d[num].to_numpy(float)
n = X.shape[0]
# TODO: sua implementação aqui
raise NotImplementedError
assert np.allclose(Sigma, np.cov(X, rowvar=False))
assert np.allclose(R, np.corrcoef(X, rowvar=False))
print(np.round(R, 3))

**1.4** Faça o histograma da massa separado por espécie. Por que a distribuição geral tem dois picos?

In [ ]:
# TODO: sua implementação aqui
raise NotImplementedError

**1.5 Paradoxo de Simpson.** Calcule a correlação entre comprimento e profundidade do bico no conjunto todo e dentro de cada espécie.

In [ ]:
# TODO: sua implementação aqui
raise NotImplementedError
print(f"todos: {r_all:.3f}"); print(r_sp.round(3))
assert r_all < 0 and (r_sp > 0).all()

## Parte 2 · Treino, teste e normalização

**2.1** Implemente `split(n, frac_teste, seed)` que devolve índices de treino e teste embaralhados. Use 25% para teste.

In [ ]:
def split(n, frac_teste=0.25, seed=0):
    # TODO: sua implementação aqui
    raise NotImplementedError

tr, te = split(len(d))
assert len(tr) + len(te) == len(d) and len(set(tr) & set(te)) == 0
print(len(tr), len(te))

**2.2** Implemente a padronização usando **apenas estatísticas do treino**: `fit_scaler` devolve $(\boldsymbol\mu, \mathbf s)$ e `transform` aplica $(\mathbf X-\mathbf 1\boldsymbol\mu^\top)\mathbf D^{-1}$.

In [ ]:
def fit_scaler(X):
    # TODO: sua implementação aqui
    raise NotImplementedError

def transform(X, mu, s):
    # TODO: sua implementação aqui
    raise NotImplementedError

mu_tr, s_tr = fit_scaler(X[tr])
Ztr = transform(X[tr], mu_tr, s_tr)
assert np.allclose(Ztr.mean(0), 0) and np.allclose(Ztr.std(0, ddof=1), 1)
print("média no teste (não é exatamente 0):", transform(X[te], mu_tr, s_tr).mean(0).round(3))

## Parte 3 · Regressão linear

Alvo: `body_mass_g`. Use o truque do bias: acrescente uma coluna de 1.

**3.1** Implemente `add_bias(X)` e `ols(X, y)` resolvendo as **equações normais** $\mathbf X^\top\mathbf X\mathbf w=\mathbf X^\top\mathbf y$ com `np.linalg.solve` (não calcule a inversa).

In [ ]:
def add_bias(X):
    # TODO: sua implementação aqui
    raise NotImplementedError

def ols(X, y):
    # TODO: sua implementação aqui
    raise NotImplementedError

x_fl = d[["flipper_length_mm"]].to_numpy(float)
y = d["body_mass_g"].to_numpy(float)
w = ols(add_bias(x_fl), y)
print("massa ≈ %.2f · nadadeira + %.1f" % (w[0], w[1]))
assert np.allclose(w, np.polyfit(x_fl[:, 0], y, 1))

**3.2 Métricas.** Implemente MSE, RMSE, MAE e $R^2$. Ajuste a reta **no treino** e avalie **no teste**.

In [ ]:
def mse(y, yh):
    # TODO: sua implementação aqui
    raise NotImplementedError

def rmse(y, yh):
    # TODO: sua implementação aqui
    raise NotImplementedError

def mae(y, yh):
    # TODO: sua implementação aqui
    raise NotImplementedError

def r2(y, yh):
    # TODO: sua implementação aqui
    raise NotImplementedError

w = ols(add_bias(x_fl[tr]), y[tr])
yh = add_bias(x_fl[te]) @ w
print(f"teste: RMSE={rmse(y[te], yh):.0f} g  MAE={mae(y[te], yh):.0f} g  R²={r2(y[te], yh):.3f}")
assert 0.6 < r2(y[te], yh) < 0.9

**3.3 Geometria.** Verifique numericamente que o resíduo do treino é ortogonal a todas as colunas de $\mathbf X$, isto é, $\mathbf X^\top(\mathbf y-\hat{\mathbf y})\approx\mathbf 0$, e que os resíduos somam zero.

In [ ]:
Xb = add_bias(x_fl[tr])
# TODO: sua implementação aqui
raise NotImplementedError
print(ortho)
assert np.allclose(ortho, 0, atol=1e-5)

**3.4 Gradiente descendente e normalização.** Implemente `gd_reg(X, y, eta, iters)` com $\nabla L=\tfrac2n\mathbf X^\top(\mathbf X\mathbf w-\mathbf y)$. Compare:
(a) nadadeira em mm, (b) nadadeira padronizada. Calcule também o número de condição de $\mathbf X^\top\mathbf X$ em cada caso e use $\eta = 1/\lambda_{\max}$ de $\tfrac2n\mathbf X^\top\mathbf X$.

In [ ]:
def gd_reg(X, y, eta, iters=500):
    w = np.zeros(X.shape[1]); hist = []
    for _ in range(iters):
        # TODO: sua implementação aqui
        raise NotImplementedError
    return w, hist

for nome, A in [("mm", add_bias(x_fl[tr])), ("padronizada", add_bias(transform(x_fl[tr], *fit_scaler(x_fl[tr]))))]:
    H = 2 / len(A) * A.T @ A
    lam = np.linalg.eigvalsh(H)
    w_gd, hist = gd_reg(A, y[tr], eta=1 / lam.max(), iters=500)
    print(f"{nome:12s} cond = {lam.max()/lam.min():12.1f}   MSE após 500 passos = {hist[-1]:12.0f}")
print("MSE mínimo (equações normais):", round(mse(y[tr], add_bias(x_fl[tr]) @ w)))

**3.5 Regressão múltipla.** Use as três medidas de bico e nadadeira **e** a espécie em one-hot (descartando uma categoria). O $R^2$ de teste melhora em relação à reta só com a nadadeira?

In [ ]:
# TODO: sua implementação aqui
raise NotImplementedError
print(f"R² teste (múltipla) = {r2_m:.3f}")
assert r2_m > r2(y[te], yh)

**3.6 Sobreajuste, ridge e validação cruzada.** Dados sintéticos $y=\sin(2\pi x)+\varepsilon$.
Implemente `ridge(Phi, y, lam)` $=(\boldsymbol\Phi^\top\boldsymbol\Phi+n\lambda\mathbf I)^{-1}\boldsymbol\Phi^\top\mathbf y$ e `kfold_mse` (validação cruzada com $k$ partes). Escolha $\lambda$ para um polinômio de grau 11.

In [ ]:
r = np.random.default_rng(7)
xs = np.sort(r.uniform(0, 1, 30)); ys = np.sin(2 * np.pi * xs) + 0.25 * r.standard_normal(30)
x_test = r.uniform(0, 1, 500); y_test = np.sin(2 * np.pi * x_test) + 0.25 * r.standard_normal(500)
poly = lambda x, p: np.vander(2 * x - 1, p + 1, increasing=True)

def ridge(Phi, y, lam):
    # TODO: sua implementação aqui
    raise NotImplementedError

def kfold_mse(x, y, p, lam, k=5, seed=0):
    idx = np.random.default_rng(seed).permutation(len(x)); folds = np.array_split(idx, k); errs = []
    for j in range(k):
        # TODO: sua implementação aqui
        raise NotImplementedError
    return np.mean(errs)

lams = 10.0 ** np.arange(-10, 1)
cv = [kfold_mse(xs, ys, 11, l) for l in lams]
best = lams[int(np.argmin(cv))]
w_best = ridge(poly(xs, 11), ys, best); w_none = ridge(poly(xs, 11), ys, 1e-12)
print("λ escolhido:", best)
print(f"MSE teste  sem regularização: {mse(y_test, poly(x_test, 11) @ w_none):.3f}   com λ*: {mse(y_test, poly(x_test, 11) @ w_best):.3f}")
plt.semilogx(lams, cv, marker="o"); plt.xlabel("λ"); plt.ylabel("MSE (validação cruzada)"); plt.show()

## Parte 4 · Classificação

### 4.1 Regressão logística do zero
Adelie (0) × Chinstrap (1), com comprimento e profundidade do bico padronizados. Implemente `sigmoid`, a entropia cruzada e o gradiente $\tfrac1n\mathbf X^\top(\mathbf p-\mathbf y)$.

In [ ]:
b2 = d[d["species"] != "Gentoo"].reset_index(drop=True)
Xl = b2[["bill_length_mm", "bill_depth_mm"]].to_numpy(float)
yl = (b2["species"] == "Chinstrap").to_numpy(float)
trl, tel = split(len(b2), seed=1)
mu_l, s_l = fit_scaler(Xl[trl])
Atr_l, Ate_l = add_bias(transform(Xl[trl], mu_l, s_l)), add_bias(transform(Xl[tel], mu_l, s_l))

def sigmoid(z):
    # TODO: sua implementação aqui
    raise NotImplementedError

def bce(y, p, eps=1e-12):
    # TODO: sua implementação aqui
    raise NotImplementedError

def logreg(X, y, eta=0.5, iters=2000):
    w = np.zeros(X.shape[1]); hist = []
    for _ in range(iters):
        # TODO: sua implementação aqui
        raise NotImplementedError
    return w, hist

wl, hist = logreg(Atr_l, yl[trl])
p_te = sigmoid(Ate_l @ wl)
acc = np.mean((p_te >= 0.5) == yl[tel])
print("w =", wl, f" acurácia teste = {acc:.3f}")
assert acc > 0.9
plt.plot(hist); plt.xlabel("passo"); plt.ylabel("entropia cruzada"); plt.show()

**4.2 Checagem do gradiente.** Compare o gradiente analítico com diferenças finitas em um $\mathbf w$ aleatório.

In [ ]:
w0 = rng.standard_normal(3); eps = 1e-6
# TODO: sua implementação aqui
raise NotImplementedError
assert np.allclose(g_an, g_num, atol=1e-6); print("gradiente OK")

### 4.3 Métricas e curva ROC
Agora um problema mais difícil: só a **nadadeira**. Implemente `confusion(y, yhat)`, as métricas, e a curva ROC variando o limiar. Calcule a AUC pela regra do trapézio.

In [ ]:
Xf = b2[["flipper_length_mm"]].to_numpy(float)
mu_f, s_f = fit_scaler(Xf[trl])
wf, _ = logreg(add_bias(transform(Xf[trl], mu_f, s_f)), yl[trl])
pf = sigmoid(add_bias(transform(Xf[tel], mu_f, s_f)) @ wf)

def confusion(y, yhat):
    # TODO: sua implementação aqui
    raise NotImplementedError

def metrics(y, yhat):
    vp, fp, fn, vn = confusion(y, yhat)
    # TODO: sua implementação aqui
    raise NotImplementedError
    return dict(acuracia=acc, precisao=prec, revocacao=rec, f1=f1)

for t in [0.3, 0.5, 0.7]:
    print(t, {k: round(float(v), 3) for k, v in metrics(yl[tel], (pf >= t).astype(float)).items()})

def roc(y, p):
    # TODO: sua implementação aqui
    raise NotImplementedError

fpr, tpr = roc(yl[tel], pf)
auc = np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2)   # regra do trapézio
print(f"AUC = {auc:.3f}")
plt.plot(fpr, tpr); plt.plot([0, 1], [0, 1], "--"); plt.xlabel("TFP"); plt.ylabel("TVP"); plt.show()

### 4.4 Regressão softmax: três espécies
$\mathbf Z=\mathbf X\mathbf W$, $\hat{\mathbf P}=\mathrm{softmax}(\mathbf Z)$ por linha, gradiente $\tfrac1n\mathbf X^\top(\hat{\mathbf P}-\mathbf Y)$. Use as quatro medidas padronizadas.

In [ ]:
classes = np.array(["Adelie", "Chinstrap", "Gentoo"])
yk = np.searchsorted(classes, d["species"].to_numpy())
Y = np.eye(3)[yk]
mu4, s4 = fit_scaler(X[tr]); A4tr, A4te = add_bias(transform(X[tr], mu4, s4)), add_bias(transform(X[te], mu4, s4))

def softmax(Z):
    # TODO: sua implementação aqui
    raise NotImplementedError

W = np.zeros((A4tr.shape[1], 3))
for _ in range(1000):
    # TODO: sua implementação aqui
    raise NotImplementedError
pred = softmax(A4te @ W).argmax(axis=1)
acc3 = np.mean(pred == yk[te])
print(f"acurácia teste (3 espécies) = {acc3:.3f}")
assert acc3 > 0.95

### 4.5 k vizinhos mais próximos
Implemente o k-NN (distância euclidiana, voto da maioria) e avalie a acurácia de teste para $k\in\{1,3,5,15,51\}$ nas medidas do **bico** padronizadas. Repita **sem padronizar** usando as quatro medidas: o que acontece e por quê?

In [ ]:
def knn_predict(Xtr, ytr, Xq, k):
    # TODO: sua implementação aqui
    raise NotImplementedError

Xb_ = d[["bill_length_mm", "bill_depth_mm"]].to_numpy(float)
mb, sb = fit_scaler(Xb_[tr])
for k in [1, 3, 5, 15, 51]:
    a_std = np.mean(knn_predict(transform(Xb_[tr], mb, sb), yk[tr], transform(Xb_[te], mb, sb), k) == yk[te])
    a_raw4 = np.mean(knn_predict(X[tr], yk[tr], X[te], k) == yk[te])
    print(f"k={k:2d}  bico padronizado: {a_std:.3f}   4 medidas sem padronizar: {a_raw4:.3f}")
print("Sem padronizar, a massa (em gramas) domina a distância e as outras medidas quase não contam.")

---
**Para casa:** compare seus resultados das partes 3.1, 4.1 e 4.4 com `sklearn.linear_model.LinearRegression`, `LogisticRegression` e `KNeighborsClassifier`. Os números batem? O que muda com a regularização padrão da `LogisticRegression`?